In [2]:
import os
import time
import re
import codecs
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
# Tạo thư mục "crawl" nếu chưa có
output_dir = "crawl"
os.makedirs(output_dir, exist_ok=True)

In [4]:
# Khởi tạo trình duyệt Chrome
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
wait = WebDriverWait(driver, 10)

In [5]:
# URL trang chủ báo Thanh Niên
homepage_url = "https://thanhnien.vn/"
driver.get(homepage_url)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))


<selenium.webdriver.remote.webelement.WebElement (session="e06d35ee1f9c96ee3d3db53ade98aaf4", element="f.716FB53BAE56ADE2A803A65893EDBA8F.d.170A0388EE367E165E1433C0C55AE977.e.97")>

In [6]:
# Lấy danh sách URL bài báo từ trang chủ
soup = BeautifulSoup(driver.page_source, "html.parser")
article_links = []

# Tìm tất cả thẻ <a> có chứa link bài viết
for link in soup.find_all("a", href=True):
    url = link["href"]
    full_url = homepage_url.rstrip("/") + url  # Ghép thành URL đầy đủ
    article_links.append(full_url)

# Loại bỏ các link trùng nhau
article_links = list(set(article_links))

print(f"🔍 Tìm thấy {len(article_links)} bài báo.")

🔍 Tìm thấy 159 bài báo.


In [ ]:
# Crawl toàn bộ bài báo
for index, article_url in enumerate(article_links):
    print(f"📌 Đang crawl bài {index + 1}: {article_url}")

    try:
        driver.get(article_url)
        time.sleep(2)  # Đợi trang tải xong

        article_soup = BeautifulSoup(driver.page_source, "html.parser")
        title = article_soup.find("h1").text.strip() if article_soup.find("h1") else "Không tìm thấy tiêu đề"
        content = ""

        # Tìm thẻ div chứa nội dung bài báo
        # Tìm thẻ div chứa nội dung bài báo
        article_div = article_soup.find("div", class_="detail-cmain")

        if article_div:
            # Lấy danh sách các thẻ <p> nhưng loại bỏ thẻ có data-placeholder="Nhập tác giả"
            content_blocks = [
                p for p in article_div.find_all("p") 
                if p.get("data-placeholder") != "Nhập tác giả"
            ]

            content = ""
            for block in content_blocks:
                # Bỏ qua nếu <p> chứa <img>
                if block.find("img"):
                    continue
                content += block.get_text(strip=True) + "\n"
        else:
            content = "Không tìm thấy nội dung bài báo."


        # Xử lý tên file hợp lệ
        file_name = re.sub(r"[^\w\s]", "", title)  # Xóa ký tự đặc biệt
        file_name = "_".join(file_name.split())[:50]  # Giới hạn 50 ký tự, thay khoảng trắng bằng _
        file_path = os.path.join(output_dir, f"{file_name}.txt")

        # Lưu nội dung bài báo vào file riêng
        with codecs.open(file_path, "w", encoding="utf-8") as file:
            file.write(content)

        print(f"✅ Lưu bài báo vào: {file_path}")

    except Exception as e:
        print(f"⚠️ Lỗi khi crawl bài {article_url}: {e}")

# Đóng trình duyệt
driver.quit()
print("✅ Crawl xong toàn bộ bài báo!")


📌 Đang crawl bài 1: https://thanhnien.vn/magazine.htm
